In [0]:
bronze_cancellation_reason = spark.table(
    "workforce_analytics.bronze.dim_cancellation_reason_raw"
)

bronze_cancellation_reason.printSchema()

display(bronze_cancellation_reason)

Step 1: Check the original Boolean values

In [0]:
display(
    bronze_cancellation_reason
    .select("is_customer_driven")
    .distinct()
)

Step 2: Create the cleaned Silver DataFrame

In [0]:
from pyspark.sql import functions as F

silver_cancellation_reason = bronze_cancellation_reason.select(
    F.trim(F.col("cancellation_id")).alias("cancellation_id"),
    F.trim(F.col("cancellation_reason")).alias("cancellation_reason"),
    F.trim(F.col("reason_group")).alias("reason_group"),
    F.lower(
        F.trim(F.col("is_customer_driven"))
    ).cast("boolean").alias("is_customer_driven"),
    F.trim(F.col("severity")).alias("severity"),
    F.col("_source_file"),
    F.col("_ingested_at")
)

Step 3: Validate the data

In [0]:
# Check schema
silver_cancellation_reason.printSchema()

# View records
display(silver_cancellation_reason)

# Check total rows
print("Total rows:", silver_cancellation_reason.count())

# Check missing cancellation IDs
print(
    "Missing cancellation IDs:",
    silver_cancellation_reason.filter(
        F.col("cancellation_id").isNull() |
        (F.col("cancellation_id") == "")
    ).count()
)

# Check duplicate IDs
duplicate_cancellation = (
    silver_cancellation_reason
    .groupBy("cancellation_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate cancellation IDs:", duplicate_cancellation.count())

display(duplicate_cancellation)

In [0]:
silver_cancellation_reason.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workforce_analytics.silver.dim_cancellation_reason")


saved_cancellation_reason = spark.table(
    "workforce_analytics.silver.dim_cancellation_reason"
)

print("Silver DimCancellationReason rows:",
      saved_cancellation_reason.count())

display(saved_cancellation_reason)

In [0]:
silver_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage",
    "dim_people",
    "dim_county",
    "dim_shrinkage_type",
    "dim_cancellation_reason"
]

for table_name in silver_tables:
    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    print(f"{table_name}: {df.count()} rows")

### complete Silver data quality assessment

In [0]:
silver_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage",
    "dim_people",
    "dim_county",
    "dim_shrinkage_type",
    "dim_cancellation_reason"
]

for table_name in silver_tables:

    table_path = f"workforce_analytics.silver.{table_name}"

    df = spark.table(table_path)

    print(
        f"{table_name}: {df.count()} rows"
    )

In [0]:
silver_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage",
    "dim_people",
    "dim_county",
    "dim_shrinkage_type",
    "dim_cancellation_reason"
]

for table_name in silver_tables:

    print("\n" + "=" * 60)
    print(f"TABLE: {table_name}")
    print("=" * 60)

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    df.printSchema()

Now let's check every column in every Silver table for:
- Null values.
- Empty strings.
- Strings containing only whitespace.

In [0]:
from pyspark.sql import functions as F

def check_nulls_and_blanks(table_name):

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    checks = []

    for column_name, data_type in df.dtypes:

        if data_type == "string":

            checks.append(
                F.sum(
                    F.when(
                        F.col(column_name).isNull() |
                        (F.trim(F.col(column_name)) == ""),
                        1
                    ).otherwise(0)
                ).alias(column_name)
            )

        else:

            checks.append(
                F.sum(
                    F.when(
                        F.col(column_name).isNull(),
                        1
                    ).otherwise(0)
                ).alias(column_name)
            )

    results = df.agg(*checks).collect()[0].asDict()

    for column_name, count in results.items():

        print(
            f"{table_name} | {column_name}: {count}"
        )


silver_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage",
    "dim_people",
    "dim_county",
    "dim_shrinkage_type",
    "dim_cancellation_reason"
]

for table_name in silver_tables:

    print("\n" + "=" * 60)
    print(f"NULL AND BLANK CHECK: {table_name}")
    print("=" * 60)

    check_nulls_and_blanks(table_name)

Step 4 — Duplicate and business-key validation
* Now we need to check whether each fact table has unique records at its correct grain.
Remember, your fact tables contain multiple snapshots. Therefore, we must check uniqueness using a combination of the record ID and snapshot_date.

In [0]:
from pyspark.sql import functions as F

key_columns = {
    "fact_work": ["work_id", "snapshot_date"],
    "fact_shift": ["shift_id", "snapshot_date"],
    "fact_shrinkage": ["shrinkage_id", "snapshot_date"],
    "dim_people": ["people_id"],
    "dim_county": ["county_id"],
    "dim_shrinkage_type": ["shrinkage_type_id"],
    "dim_cancellation_reason": ["cancellation_id"]
}

for table_name, keys in key_columns.items():

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    duplicate_records = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
    )

    print(
        f"{table_name}: "
        f"{duplicate_records.count()} duplicate keys"
    )

    duplicate_records.show()

Step 5 — Business-rule and date validation
Next, we'll check whether the actual values make business sense.
For example:
- A scheduled end time should be later than its start time.
- A shrinkage end time should be later than its start time.
- Snapshot dates should match the dates encoded in the source filenames.
- Status values should be consistent.

In [0]:
from pyspark.sql import functions as F

# Fact Work: invalid scheduled time ranges
work_invalid_times = spark.table(
    "workforce_analytics.silver.fact_work"
).filter(
    F.col("scheduled_end") <= F.col("scheduled_start")
)

print(
    "Fact Work invalid time ranges:",
    work_invalid_times.count()
)

display(work_invalid_times.limit(10))


# Fact Shift: invalid scheduled time ranges
shift_invalid_times = spark.table(
    "workforce_analytics.silver.fact_shift"
).filter(
    F.col("scheduled_end") <= F.col("scheduled_start")
)

print(
    "Fact Shift invalid time ranges:",
    shift_invalid_times.count()
)

display(shift_invalid_times.limit(10))


# Fact Shrinkage: invalid shrinkage time ranges
shrinkage_invalid_times = spark.table(
    "workforce_analytics.silver.fact_shrinkage"
).filter(
    F.col("end_datetime") <= F.col("start_datetime")
)

print(
    "Fact Shrinkage invalid time ranges:",
    shrinkage_invalid_times.count()
)

display(shrinkage_invalid_times.limit(10))

Step 6 — Snapshot consistency
* This is particularly important for your project because your three fact tables contain five daily snapshots.
We want to confirm:
1. All five snapshot dates exist in each fact table.
2. The records are distributed correctly across the snapshots.
3. The snapshot dates agree with the dates in the source filenames

In [0]:
from pyspark.sql import functions as F

fact_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage"
]

for table_name in fact_tables:

    print("\n" + "=" * 50)
    print(f"SNAPSHOT VALIDATION: {table_name}")
    print("=" * 50)

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    display(
        df.groupBy("snapshot_date")
          .agg(
              F.count("*").alias("row_count"),
              F.countDistinct(
                  "_source_file"
              ).alias("source_file_count")
          )
          .orderBy("snapshot_date")
    )

Step 7 — Referential integrity
* Now we'll check whether the foreign keys in your fact tables have matching records in the dimensions.
* Your expected relationships are:

Fact Work
- fact_work.people_id → dim_people.people_id
- fact_work.county_id → dim_county.county_id
- fact_work.cancellation_id → dim_cancellation_reason.cancellation_id

Fact Shift
- fact_shift.people_id → dim_people.people_id
- fact_shift.county_id → dim_county.county_id

Fact Shrinkage
- fact_shrinkage.people_id → dim_people.people_id
- fact_shrinkage.shrinkage_type_id → dim_shrinkage_type.shrinkage_type_id

In [0]:
from pyspark.sql import functions as F

relationships = [
    ("fact_work", "people_id", "dim_people", "people_id"),
    ("fact_work", "county_id", "dim_county", "county_id"),
    ("fact_work", "cancellation_id", "dim_cancellation_reason", "cancellation_id"),

    ("fact_shift", "people_id", "dim_people", "people_id"),
    ("fact_shift", "county_id", "dim_county", "county_id"),

    ("fact_shrinkage", "people_id", "dim_people", "people_id"),
    ("fact_shrinkage", "shrinkage_type_id", "dim_shrinkage_type", "shrinkage_type_id")
]

for fact_table, fact_key, dim_table, dim_key in relationships:

    fact_df = spark.table(
        f"workforce_analytics.silver.{fact_table}"
    )

    dim_df = spark.table(
        f"workforce_analytics.silver.{dim_table}"
    )

    unmatched = (
        fact_df.alias("f")
        .join(
            dim_df.alias("d"),
            F.col(f"f.{fact_key}") == F.col(f"d.{dim_key}"),
            "left_anti"
        )
    )

    print(
        f"{fact_table}.{fact_key} -> "
        f"{dim_table}.{dim_key}: "
        f"{unmatched.count()} unmatched records"
    )

    display(unmatched.limit(5))

* This means every foreign key in your fact tables has a corresponding record in its dimension table.

In practical terms:
- Every person referenced in a fact table exists in dim_people.
- Every county exists in dim_county.
- Every cancellation reason exists in dim_cancellation_reason.
- Every shrinkage type exists in dim_shrinkage_type.

In [0]:
categorical_columns = {
    "fact_work": ["status"],
    "dim_people": ["employment_type", "job_role"],
    "dim_county": ["urban_rural"],
    "dim_shrinkage_type": ["category"],
    "dim_cancellation_reason": ["severity", "reason_group"]
}

for table_name, columns in categorical_columns.items():

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    for column_name in columns:

        print(
            f"\n{table_name}.{column_name}"
        )

        display(
            df.groupBy(column_name)
              .count()
              .orderBy(column_name)
        )

Step 9 — Check date and snapshot consistency
* We should now verify that the snapshot date extracted from the filename agrees with the actual _source_file value.

In [0]:
from pyspark.sql import functions as F

fact_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage"
]

for table_name in fact_tables:

    df = spark.table(
        f"workforce_analytics.silver.{table_name}"
    )

    checked_df = df.withColumn(
        "filename_date",
        F.to_date(
            F.regexp_extract(
                F.col("_source_file"),
                r"(\d{2}_\d{2}_\d{4})",
                1
            ),
            "dd_MM_yyyy"
        )
    )

    mismatches = checked_df.filter(
        F.col("snapshot_date") != F.col("filename_date")
    )

    print(
        f"{table_name}: "
        f"{mismatches.count()} snapshot date mismatches"
    )

    display(
        mismatches.select(
            "_source_file",
            "snapshot_date",
            "filename_date"
        ).limit(10)
    )